# Lab 2 · Del dato crudo al agente comercial multipaís
### ISM Snow Day · Equipo técnico · 90 minutos · Cortex Code + AI Functions + Semantic Views + Cortex Agents

**El caso.** En el cierre del mes, la venta neta de **Rayo Energy Lata 473ml cayó 47% en ISM Dominicana**. El tablero de ventas dice *cuánto* cayó, pero no *por qué*. El porqué está escrito en **las notas de visita de los vendedores**: miles de comentarios en texto libre, en español y en francés (Haití), que nadie alcanza a leer.

**Su misión.** Construir, conversando con **Cortex Code**, el flujo completo que convierte esas notas en respuestas:

| # | Sección | Qué construye | Min |
|---|---|---|---|
| 1 | Preparación | Datos sintéticos de ventas, inventario y 1,200 notas de visita | 5 |
| 2 | Explorar con Cortex Code | Confirmar la caída y leer las notas | 10 |
| 3 | Prototipo de IA | `AI_TRANSLATE`, `AI_CLASSIFY`, `AI_SENTIMENT`, `AI_EXTRACT`, `AI_FILTER` sobre 10 notas | 10 |
| 4 | Pipeline de IA | Una **Dynamic Table incremental** que enriquece todas las notas | 8 |
| 5 | Validar la IA | Medir errores reales de la IA y corregirlos (v2) | 15 |
| 6 | Incremental | Llega un lote nuevo: la IA procesa solo lo nuevo | 7 |
| 7 | Modelo semántico | La cifra de ventas que no cuadra con Finanzas, y cómo corregirla | 12 |
| 8 | Agente + CoWork | El **Asistente Comercial ISM**, publicado para todos | 10 |
| 9 | Evaluación multipaís | Probar al agente con preguntas de varios países e idiomas | 8 |
| ★ | Reto opcional | Días de cobertura de inventario, para quienes terminen antes | — |

> **Datos 100% sintéticos.** Las tablas usan la estructura de ISM (sociedades SAP, ventas, MB52), con marcas, clientes y notas ficticias.

### Cómo trabajar con Cortex Code en este lab

Abra el panel de **Cortex Code** (ícono ✦ en la esquina inferior derecha del notebook). En cada sección encontrará un **💬 Prompt** para copiar. Cortex Code lee su notebook y sus tablas, escribe el SQL y se lo propone en una celda nueva.

Cada sección también trae una **celda de referencia** ya escrita: si Cortex Code le propone algo distinto, compare las dos. Así aprende qué pedirle y cómo revisar lo que propone.

Hábitos que funcionan:
- **Dé contexto de negocio**, no solo nombres de tablas: *"la venta oficial es la venta neta"*.
- **Pida explicaciones**: *"explícame esta consulta línea por línea"*.
- **Pida validación**: *"¿cómo sé que la IA acertó?"*.

## 1 · Preparación (5 min)
Ejecute las tres celdas. Crean su warehouse, su schema `ISM_SNOWDAY.LAB2` y los datos.

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

# Usted es dueño de esta cuenta (ACCOUNTADMIN), así que el lab crea todo lo que necesita.
session.sql("USE ROLE ACCOUNTADMIN").collect()

db = "ISM_SNOWDAY"
schema = "LAB2"
fq = f"{db}.{schema}"
wh = "ISM_SNOWDAY_WH"

for stmt in [
    # Warehouse pequeño para las consultas, las funciones de IA y el agente
    f"CREATE WAREHOUSE IF NOT EXISTS {wh} WAREHOUSE_SIZE = XSMALL AUTO_SUSPEND = 60 AUTO_RESUME = TRUE",
    f"USE WAREHOUSE {wh}",
    # Permite usar modelos de IA de otra región si alguno no está disponible en la región de esta cuenta
    "ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'ANY_REGION'",
    f"CREATE DATABASE IF NOT EXISTS {db}",
    f"CREATE SCHEMA IF NOT EXISTS {fq}",
    f"USE SCHEMA {fq}",
]:
    session.sql(stmt).collect()

print(f"Rol        : {session.get_current_role()}")
print(f"Su schema  : {fq}")
print(f"Warehouse  : {wh}")

In [ ]:
# Genera los datos sintéticos de ISM: mismas tablas y columnas que usa ISM (estilo SAP), con marcas y clientes ficticios.
# No necesita leer esta celda: solo ejecútela (tarda alrededor de 1 minuto).

SOCIEDADES = [
    # id, bukrs, nombre, id_pais, pais, moneda, fx (moneda por USD), peso
    (1000, "DO01", "ISM Dominicana", "DO", "República Dominicana", "DOP", 58.9, 0.42),
    (2000, "GT01", "ISM Guatemala", "GT", "Guatemala", "GTQ", 7.75, 0.22),
    (3000, "HT01", "ISM Haití", "HT", "Haití", "HTG", 132.0, 0.10),
    (4000, "PE01", "ISM Perú", "PE", "Perú", "PEN", 3.74, 0.18),
    (5000, "CL01", "ISM Chile", "CL", "Chile", "CLP", 935.0, 0.08),
]

PRODUCTOS = [
    # sku, producto, marca, categoria, formato, litros_por_caja, precio_caja_usd, costo_caja_usd
    ("PT1001", "Kola Isla Lata 355ml", "Kola Isla", "Gaseosas", "355ml", 8.52, 9.60, 5.30),
    ("PT1002", "Kola Isla 600ml", "Kola Isla", "Gaseosas", "600ml", 14.40, 10.80, 5.90),
    ("PT1003", "Kola Isla 1.5L", "Kola Isla", "Gaseosas", "1.5L", 18.00, 11.40, 6.80),
    ("PT1004", "Kola Isla 2L", "Kola Isla", "Gaseosas", "2L", 16.00, 9.90, 6.10),
    ("PT1005", "Kola Isla 3L", "Kola Isla", "Gaseosas", "3L", 18.00, 10.20, 6.60),
    ("PT1101", "Fresquita Naranja 600ml", "Fresquita", "Gaseosas", "600ml", 14.40, 10.20, 5.70),
    ("PT1102", "Fresquita Naranja 2L", "Fresquita", "Gaseosas", "2L", 16.00, 9.30, 5.90),
    ("PT1103", "Fresquita Uva 600ml", "Fresquita", "Gaseosas", "600ml", 14.40, 10.20, 5.70),
    ("PT1104", "Fresquita Uva 2L", "Fresquita", "Gaseosas", "2L", 16.00, 9.30, 5.90),
    ("PT1105", "Fresquita Limón 600ml", "Fresquita", "Gaseosas", "600ml", 14.40, 10.20, 5.70),
    ("PT1106", "Fresquita Limón 2L", "Fresquita", "Gaseosas", "2L", 16.00, 9.30, 5.90),
    ("PT2001", "Agua Montaña 600ml", "Agua Montaña", "Agua", "600ml", 14.40, 6.00, 2.60),
    ("PT2002", "Agua Montaña 1L", "Agua Montaña", "Agua", "1L", 12.00, 5.40, 2.40),
    ("PT2003", "Agua Montaña 5L", "Agua Montaña", "Agua", "5L", 20.00, 4.80, 2.30),
    ("PT2004", "Agua Montaña con Gas 600ml", "Agua Montaña", "Agua", "600ml", 14.40, 7.20, 3.20),
    ("PT3001", "Rayo Energy Lata 250ml", "Rayo", "Energizantes", "250ml", 6.00, 16.80, 8.10),
    ("PT3002", "Rayo Energy Lata 473ml", "Rayo", "Energizantes", "473ml", 11.35, 25.20, 11.90),
    ("PT3003", "Rayo Zero Lata 473ml", "Rayo", "Energizantes", "473ml", 11.35, 25.20, 12.20),
    ("PT4001", "Frutalia Naranja 1L", "Frutalia", "Jugos y Néctares", "1L", 12.00, 13.20, 7.40),
    ("PT4002", "Frutalia Durazno 250ml", "Frutalia", "Jugos y Néctares", "250ml", 6.00, 9.60, 5.40),
    ("PT4003", "Frutalia Mango 1L", "Frutalia", "Jugos y Néctares", "1L", 12.00, 13.20, 7.50),
    ("PT5001", "Hidra+ Mora 500ml", "Hidra+", "Isotónicos", "500ml", 6.00, 9.00, 4.40),
    ("PT5002", "Hidra+ Limón 500ml", "Hidra+", "Isotónicos", "500ml", 6.00, 9.00, 4.40),
    ("PT5003", "Hidra+ Tropical 1L", "Hidra+", "Isotónicos", "1L", 12.00, 11.40, 5.80),
    ("PT6001", "Té Brisa Limón 500ml", "Té Brisa", "Té", "500ml", 6.00, 8.40, 4.30),
    ("PT6002", "Té Brisa Durazno 500ml", "Té Brisa", "Té", "500ml", 6.00, 8.40, 4.30),
]

# Non-finished-goods materials for the MB52 inventory snapshot: material, descripcion, tipo, unidad, costo_usd, cantidad_base
MATERIALES = [
    ("MP0001", "Azúcar refinada", "Materia prima", "KG", 0.62, 180000),
    ("MP0002", "Concentrado sabor cola", "Materia prima", "L", 21.50, 2600),
    ("MP0003", "Concentrado sabor frutas", "Materia prima", "L", 18.40, 2100),
    ("MP0004", "CO2 grado alimenticio", "Materia prima", "KG", 0.48, 52000),
    ("MP0005", "Pulpa de fruta", "Materia prima", "KG", 1.35, 24000),
    ("MP0006", "Cafeína y taurina (premezcla)", "Materia prima", "KG", 38.00, 420),
    ("EN0001", "Preforma PET 23g", "Envases y empaques", "UN", 0.031, 2600000),
    ("EN0002", "Preforma PET 48g", "Envases y empaques", "UN", 0.058, 900000),
    ("EN0003", "Tapa plástica 28mm", "Envases y empaques", "UN", 0.006, 3800000),
    ("EN0004", "Lata de aluminio 473ml", "Envases y empaques", "UN", 0.115, 640000),
    ("EN0005", "Etiqueta termoencogible", "Envases y empaques", "UN", 0.009, 3100000),
    ("EN0006", "Caja de cartón corrugado", "Envases y empaques", "UN", 0.42, 160000),
    ("RP0001", "Rodamientos línea de llenado", "Repuestos", "UN", 46.00, 310),
    ("RP0002", "Válvula llenadora", "Repuestos", "UN", 128.00, 140),
    ("RP0003", "Banda transportadora", "Repuestos", "UN", 410.00, 36),
    ("PR0001", "Hielera con logo", "Material promocional", "UN", 22.00, 1800),
    ("PR0002", "Exhibidor de punto de venta", "Material promocional", "UN", 64.00, 650),
]


def _values(rows):
    out = []
    for r in rows:
        vals = []
        for v in r:
            if isinstance(v, str):
                vals.append("'" + v.replace("'", "''") + "'")
            else:
                vals.append(str(v))
        out.append("(" + ", ".join(vals) + ")")
    return ",\n".join(out)


def load_statements(fq: str, n_clientes: int = 700):
    """SQL statements that (re)build the shared ISM model in schema fq."""
    soc = _values(SOCIEDADES)
    prod = _values(PRODUCTOS)
    mat = _values(MATERIALES)
    return [
        # ---------- Dimensions ----------
        f"""CREATE OR REPLACE TABLE {fq}.DIM_SOCIEDAD (
              ID_SOCIEDAD NUMBER, BUKRS_SOCIEDAD STRING, SOCIEDAD STRING, ID_PAIS STRING, PAIS STRING,
              MONEDA STRING, TIPO_CAMBIO_USD NUMBER(12,4), PESO_NEGOCIO NUMBER(5,2))
            COMMENT = 'Sociedades (company codes SAP) de ISM. Datos sintéticos.'
            AS SELECT * FROM VALUES {soc}""",
        f"""CREATE OR REPLACE TABLE {fq}.DIM_PRODUCTO (
              ID_SKU_PRODUCTO STRING, PRODUCTO STRING, MARCA STRING, CATEGORIA STRING, FORMATO STRING,
              LITROS_POR_CAJA NUMBER(8,2), PRECIO_LISTA_CAJA_USD NUMBER(10,2), COSTO_ESTANDAR_CAJA_USD NUMBER(10,2))
            COMMENT = 'Maestro de producto terminado (SKU). Marcas ficticias.'
            AS SELECT * FROM VALUES {prod}""",
        f"""CREATE OR REPLACE TABLE {fq}.DIM_CLIENTE COMMENT = 'Maestro de clientes por sociedad y canal. Datos sintéticos.' AS
            WITH g AS (SELECT SEQ4() + 1 AS n FROM TABLE(GENERATOR(ROWCOUNT => {n_clientes}))),
            b AS (
              SELECT n,
                     MOD(ABS(HASH(n, 'soc')), 100) AS hs,
                     MOD(ABS(HASH(n, 'canal')), 100) AS hc,
                     MOD(ABS(HASH(n, 'nom')), 12) AS hn,
                     MOD(ABS(HASH(n, 'ciu')), 4) AS hci,
                     0.4 + MOD(ABS(HASH(n, 'tam')), 160) / 100 AS tam
              FROM g)
            SELECT 'C' || LPAD(n, 6, '0') AS ID_CUENTA,
                   CASE WHEN hs < 42 THEN 1000 WHEN hs < 64 THEN 2000 WHEN hs < 74 THEN 3000 WHEN hs < 92 THEN 4000 ELSE 5000 END AS ID_SOCIEDAD,
                   CASE WHEN hc < 62 THEN 'Tradicional' WHEN hc < 74 THEN 'Moderno' WHEN hc < 88 THEN 'Mayorista' ELSE 'On Premise' END AS CANAL_CORPORATIVO,
                   CASE WHEN hc < 62 THEN IFF(ID_SOCIEDAD = 1000, 'Colmado ', 'Tienda ') || ARRAY_CONSTRUCT('La Esperanza','El Progreso','Don Pedro','La Bendición','Los Hermanos','El Buen Precio','La Fe','San José','Doña María','El Sol','La Económica','Mi Barrio')[hn]::STRING
                        WHEN hc < 74 THEN ARRAY_CONSTRUCT('Supermercados Del Mar','Hiper Ahorro','Mercado Familiar','Super Express','Cadena Uno','Supermercado La Plaza','Mega Compras','Super Fresco','Plaza Market','Mercado Central','Super Norte','Hiper Sur')[hn]::STRING
                        WHEN hc < 88 THEN 'Distribuidora ' || ARRAY_CONSTRUCT('Del Norte','Caribe','Andina','Central','El Puerto','Del Valle Alto','La Unión','San Miguel Norte','El Cruce','Oriental','Los Pinos','Mercantil')[hn]::STRING
                        ELSE ARRAY_CONSTRUCT('Restaurante El Fogón','Bar La Terraza','Hotel Costa Azul','Cafetería Central','Pica Pollo Don Juan','Bar El Muelle','Hotel Plaza','Restaurante Sabores','Food Court Ágora','Cafetería La Esquina','Bar Los Amigos','Hotel del Parque')[hn]::STRING END
                   || ' ' || LPAD(n, 3, '0') AS NOMBRE_CLIENTE,
                   CASE ID_SOCIEDAD
                     WHEN 1000 THEN ARRAY_CONSTRUCT('Santo Domingo Oeste','Santo Domingo Este','Santiago','San Cristóbal')[hci]
                     WHEN 2000 THEN ARRAY_CONSTRUCT('Ciudad de Guatemala','Mixco','Quetzaltenango','Escuintla')[hci]
                     WHEN 3000 THEN ARRAY_CONSTRUCT('Puerto Príncipe','Cap-Haïtien','Pétion-Ville','Les Cayes')[hci]
                     WHEN 4000 THEN ARRAY_CONSTRUCT('Lima','Arequipa','Trujillo','Piura')[hci]
                     ELSE ARRAY_CONSTRUCT('Santiago de Chile','Valparaíso','Concepción','Antofagasta')[hci] END::STRING AS CIUDAD,
                   ROUND(tam * CASE WHEN hc < 62 THEN 1 WHEN hc < 74 THEN 9 WHEN hc < 88 THEN 16 ELSE 1.6 END, 2) AS TAMANO_RELATIVO
            FROM b
            UNION ALL
            SELECT 'C900001', 1000, 'Moderno', 'Hiper Ahorro Santo Domingo (cuenta clave)', 'Santo Domingo Oeste', 30""",
        # ---------- Sales fact (monthly, by client x SKU) ----------
        f"""CREATE OR REPLACE TABLE {fq}.FACT_DETALLE_VENTAS COMMENT = 'Detalle de ventas mensual por cliente y SKU (USD y moneda local). Datos sintéticos.' AS
            WITH meses AS (
              SELECT DATEADD(month, -(SEQ4()), DATE_TRUNC('month', CURRENT_DATE())) AS PERIODO, SEQ4() AS meses_atras
              FROM TABLE(GENERATOR(ROWCOUNT => 18))),
            surtido AS (
              SELECT c.*, p.*
              FROM {fq}.DIM_CLIENTE c CROSS JOIN {fq}.DIM_PRODUCTO p
              WHERE MOD(ABS(HASH(c.ID_CUENTA, p.ID_SKU_PRODUCTO)), 100) <
                    CASE c.CANAL_CORPORATIVO WHEN 'Moderno' THEN 85 WHEN 'Mayorista' THEN 70 WHEN 'On Premise' THEN 30 ELSE 32 END),
            base AS (
              SELECT s.ID_CUENTA, s.ID_SOCIEDAD, s.CANAL_CORPORATIVO, s.ID_SKU_PRODUCTO, s.CATEGORIA,
                     s.PRECIO_LISTA_CAJA_USD, s.COSTO_ESTANDAR_CAJA_USD, s.LITROS_POR_CAJA, m.PERIODO, m.meses_atras,
                     GREATEST(1, ROUND(
                       CASE s.CANAL_CORPORATIVO WHEN 'Moderno' THEN 56 WHEN 'Mayorista' THEN 72 WHEN 'On Premise' THEN 40 ELSE 48 END
                       * s.TAMANO_RELATIVO
                       * (1 + 0.12 * SIN((MONTH(m.PERIODO) - 3) * 3.14159 / 6))
                       * (1 - 0.006 * m.meses_atras)
                       * (0.75 + MOD(ABS(HASH(s.ID_CUENTA, s.ID_SKU_PRODUCTO, m.PERIODO)), 50) / 100)
                       * IFF(m.meses_atras = 0 AND s.ID_SKU_PRODUCTO = 'PT3002', 0.55, 1)
                       * IFF(m.meses_atras = 0 AND s.ID_SKU_PRODUCTO = 'PT4001', 0.72, 1)
                       * IFF(m.meses_atras = 0 AND s.ID_CUENTA = 'C900001', 0.42, 1)
                     )) AS CANTIDAD_CAJAS
              FROM surtido s CROSS JOIN meses m),
            calc AS (
              SELECT b.*,
                     ROUND(CANTIDAD_CAJAS * PRECIO_LISTA_CAJA_USD * (1 + 0.002 * (17 - meses_atras)), 2) AS USD_VALOR_BRUTO,
                     (CASE CANAL_CORPORATIVO WHEN 'Moderno' THEN 0.09 WHEN 'Mayorista' THEN 0.11 WHEN 'On Premise' THEN 0.03 ELSE 0.04 END
                       + MOD(ABS(HASH(ID_CUENTA, PERIODO, 'd')), 30) / 1000) AS pct_desc,
                     (CASE CANAL_CORPORATIVO WHEN 'Moderno' THEN 0.025 WHEN 'Mayorista' THEN 0.04 WHEN 'On Premise' THEN 0.05 ELSE 0.06 END
                       + IFF(meses_atras = 0 AND ID_SOCIEDAD = 1000, 0.025, 0)
                       + MOD(ABS(HASH(ID_CUENTA, PERIODO, 'b')), 20) / 1000) AS pct_bonif,
                     (0.008 + MOD(ABS(HASH(ID_CUENTA, ID_SKU_PRODUCTO, PERIODO, 'r')), 25) / 1000) AS pct_rech,
                     (1 + IFF(meses_atras <= 2, 0.04, 0) + (MOD(ABS(HASH(ID_SKU_PRODUCTO, PERIODO, 'c')), 40) - 20) / 1000) AS f_costo
              FROM base b)
            SELECT 'V' || LPAD(ROW_NUMBER() OVER (ORDER BY PERIODO, ID_CUENTA, ID_SKU_PRODUCTO), 8, '0') AS ID_VENTA,
                   DATEADD(day, MOD(ABS(HASH(ID_CUENTA, ID_SKU_PRODUCTO, PERIODO, 'f')), IFF(PERIODO = DATE_TRUNC('month', CURRENT_DATE()), DAY(CURRENT_DATE()), 28)), PERIODO) AS FECHA,
                   PERIODO, c.ID_SOCIEDAD, ID_CUENTA, ID_SKU_PRODUCTO, CANAL_CORPORATIVO, CATEGORIA AS CATEGORIA_VENTA,
                   CANTIDAD_CAJAS, ROUND(CANTIDAD_CAJAS * LITROS_POR_CAJA, 2) AS LITROS,
                   USD_VALOR_BRUTO,
                   ROUND(USD_VALOR_BRUTO * pct_desc, 2) AS USD_DESCUENTOS,
                   ROUND(USD_VALOR_BRUTO * pct_bonif, 2) AS USD_BONIFICACIONES,
                   ROUND(USD_VALOR_BRUTO * pct_rech, 2) AS USD_VALOR_RECHAZO,
                   ROUND(USD_VALOR_BRUTO * (1 - pct_desc - pct_bonif - pct_rech), 2) AS USD_VALOR_NETO,
                   ROUND(CANTIDAD_CAJAS * (1 - pct_rech) * COSTO_ESTANDAR_CAJA_USD * f_costo, 2) AS USD_COSTO_VENTA,
                   s.MONEDA,
                   ROUND(USD_VALOR_BRUTO * (1 - pct_desc - pct_bonif - pct_rech) * s.TIPO_CAMBIO_USD, 2) AS VALOR_NETO_ML
            FROM calc c JOIN {fq}.DIM_SOCIEDAD s ON s.ID_SOCIEDAD = c.ID_SOCIEDAD""",
        # ---------- Monthly P&L by company (estado de resultados) ----------
        f"""CREATE OR REPLACE TABLE {fq}.ESTADO_RESULTADOS_MENSUAL COMMENT = 'Estado de resultados mensual por sociedad (USD). Montos con signo: ingresos positivos, costos y gastos negativos.' AS
            WITH v AS (
              SELECT PERIODO, ID_SOCIEDAD, SUM(USD_VALOR_NETO) AS neto, SUM(USD_COSTO_VENTA) AS costo
              FROM {fq}.FACT_DETALLE_VENTAS GROUP BY 1, 2),
            l AS (
              SELECT * FROM VALUES
                (1, 'Ventas netas', 'Ingreso', TRUE, 0.0),
                (2, 'Costo de ventas', 'Costo', TRUE, 0.0),
                (3, 'Gastos de venta y mercadeo', 'Gasto operativo', TRUE, 0.095),
                (4, 'Gastos de distribución', 'Gasto operativo', TRUE, 0.078),
                (5, 'Gastos administrativos', 'Gasto operativo', TRUE, 0.052),
                (6, 'Depreciación y amortización', 'Depreciación', FALSE, 0.031),
                (7, 'Gastos financieros', 'Financiero', FALSE, 0.012)
                AS l(ORDEN, LINEA_RESULTADO, TIPO_LINEA, INCLUIDO_EN_EBITDA, pct)),
            pre AS (
              SELECT v.PERIODO, v.ID_SOCIEDAD, l.ORDEN, l.LINEA_RESULTADO, l.TIPO_LINEA, l.INCLUIDO_EN_EBITDA,
                     ROUND(CASE l.ORDEN WHEN 1 THEN v.neto WHEN 2 THEN -v.costo
                           ELSE -v.neto * l.pct * (0.92 + MOD(ABS(HASH(v.PERIODO, v.ID_SOCIEDAD, l.ORDEN)), 16) / 100)
                                 * IFF(l.ORDEN = 4 AND v.ID_SOCIEDAD = 3000, 1.35, 1) END, 2) AS USD_MONTO
              FROM v CROSS JOIN l)
            SELECT * FROM pre
            UNION ALL
            SELECT PERIODO, ID_SOCIEDAD, 8, 'Impuesto sobre la renta', 'Impuesto', FALSE,
                   ROUND(-GREATEST(SUM(USD_MONTO), 0) * 0.27, 2)
            FROM pre GROUP BY PERIODO, ID_SOCIEDAD""",
        # ---------- MB52-style inventory snapshot (month-end) ----------
        f"""CREATE OR REPLACE TABLE {fq}.INVENTARIOS_MB52 COMMENT = 'Foto de inventario a fin de mes por sociedad, centro y material (estilo SAP MB52). Valores en USD.' AS
            WITH meses AS (
              SELECT DATEADD(month, -(SEQ4()), DATE_TRUNC('month', CURRENT_DATE())) AS FECHA_PERIODO, SEQ4() AS meses_atras
              FROM TABLE(GENERATOR(ROWCOUNT => 18))),
            mat AS (
              SELECT ID_SKU_PRODUCTO AS MATERIAL, PRODUCTO AS DESCRIPCION, 'Producto terminado' AS TIPO_MATERIAL, 'CJ' AS UBM,
                     COSTO_ESTANDAR_CAJA_USD AS COSTO_USD, 9000 AS BASE
              FROM {fq}.DIM_PRODUCTO
              UNION ALL
              SELECT * FROM VALUES {mat} AS m(MATERIAL, DESCRIPCION, TIPO_MATERIAL, UBM, COSTO_USD, BASE)),
            x AS (
              SELECT m.FECHA_PERIODO, m.meses_atras, s.ID_SOCIEDAD, s.SOCIEDAD, s.ID_PAIS, s.PESO_NEGOCIO, mat.*,
                     ROUND(8 * mat.BASE * s.PESO_NEGOCIO * (0.6 + MOD(ABS(HASH(m.FECHA_PERIODO, s.ID_SOCIEDAD, mat.MATERIAL)), 80) / 100)
                           * IFF(m.meses_atras = 0 AND mat.TIPO_MATERIAL = 'Materia prima' AND s.ID_SOCIEDAD = 1000, 1.45, 1)) AS STOCK_LIB_UTIL,
                     ROUND(8 * mat.BASE * s.PESO_NEGOCIO * MOD(ABS(HASH(m.FECHA_PERIODO, s.ID_SOCIEDAD, mat.MATERIAL, 't')), 18) / 100) AS STOCK_TRANSITO,
                     ROUND(8 * mat.BASE * s.PESO_NEGOCIO * MOD(ABS(HASH(m.FECHA_PERIODO, s.ID_SOCIEDAD, mat.MATERIAL, 'b')), 4) / 100) AS STOCK_BLOQ
              FROM meses m CROSS JOIN {fq}.DIM_SOCIEDAD s CROSS JOIN mat)
            SELECT LEAST(LAST_DAY(FECHA_PERIODO), CURRENT_DATE()) AS FECHA_PERIODO, ID_SOCIEDAD AS SOCIEDAD, SOCIEDAD AS NOMBRE_SOCIEDAD, ID_PAIS AS PAIS,
                   ID_PAIS || '01' AS CENTRO, IFF(TIPO_MATERIAL = 'Producto terminado', 'PT01', 'MP01') AS ALMACEN,
                   MATERIAL, DESCRIPCION, TIPO_MATERIAL, UBM,
                   STOCK_LIB_UTIL, ROUND(STOCK_LIB_UTIL * COSTO_USD, 2) AS VALOR_LIB_UTIL,
                   STOCK_TRANSITO, ROUND(STOCK_TRANSITO * COSTO_USD, 2) AS VALOR_TRANSITO,
                   STOCK_BLOQ, ROUND(STOCK_BLOQ * COSTO_USD, 2) AS VALOR_STOCK_BLOQ,
                   ROUND((STOCK_LIB_UTIL + STOCK_TRANSITO + STOCK_BLOQ) * COSTO_USD, 2) AS VALOR_INVENTARIO_TOTAL,
                   (meses_atras = 0) AS ES_ULTIMO_CORTE
            FROM x""",
        # ---------- Cash flow (treasury movements) ----------
        f"""CREATE OR REPLACE TABLE {fq}.FLUJO_EFECTIVO COMMENT = 'Movimientos de tesorería: entradas y salidas de efectivo por concepto (USD y moneda local). Datos sintéticos.' AS
            WITH v AS (
              SELECT PERIODO, ID_SOCIEDAD, SUM(USD_VALOR_NETO) AS neto, SUM(USD_COSTO_VENTA) AS costo
              FROM {fq}.FACT_DETALLE_VENTAS GROUP BY 1, 2),
            conceptos AS (
              SELECT * FROM VALUES
                ('Entrada', 'Cobranza a clientes', 'neto', 0.97, 8),
                ('Entrada', 'Otros ingresos', 'neto', 0.012, 2),
                ('Entrada', 'Desembolso de préstamos', 'neto', 0.06, 1),
                ('Salida', 'Pago a proveedores', 'costo', 0.93, 6),
                ('Salida', 'Nómina y prestaciones', 'neto', 0.085, 2),
                ('Salida', 'Impuestos', 'neto', 0.07, 1),
                ('Salida', 'Fletes y distribución', 'neto', 0.06, 4),
                ('Salida', 'Servicios y mantenimiento', 'neto', 0.022, 3),
                ('Salida', 'Inversiones (CAPEX)', 'neto', 0.045, 2),
                ('Salida', 'Pago de deuda e intereses', 'neto', 0.035, 1)
                AS c(TIPO_MOVIMIENTO, CONCEPTO, base, pct, n_mov)),
            k AS (SELECT SEQ4() + 1 AS k FROM TABLE(GENERATOR(ROWCOUNT => 8))),
            x AS (
              SELECT v.PERIODO, v.ID_SOCIEDAD, c.TIPO_MOVIMIENTO, c.CONCEPTO, k.k, c.n_mov,
                     IFF(c.base = 'neto', v.neto, v.costo) * c.pct
                       * (0.85 + MOD(ABS(HASH(v.PERIODO, v.ID_SOCIEDAD, c.CONCEPTO)), 30) / 100)
                       * IFF(c.CONCEPTO = 'Inversiones (CAPEX)' AND v.PERIODO = DATE_TRUNC('month', CURRENT_DATE()) AND v.ID_SOCIEDAD = 1000, 3.2, 1)
                       / c.n_mov AS monto
              FROM v CROSS JOIN conceptos c JOIN k ON k.k <= c.n_mov
              WHERE NOT (c.CONCEPTO = 'Desembolso de préstamos' AND MOD(ABS(HASH(v.PERIODO, v.ID_SOCIEDAD)), 5) <> 0))
            SELECT 'M' || LPAD(ROW_NUMBER() OVER (ORDER BY x.PERIODO, x.ID_SOCIEDAD, CONCEPTO, k), 7, '0') AS ID_MOVIMIENTO,
                   DATEADD(day, MOD(ABS(HASH(x.PERIODO, x.ID_SOCIEDAD, CONCEPTO, k)), IFF(x.PERIODO = DATE_TRUNC('month', CURRENT_DATE()), DAY(CURRENT_DATE()), 28)), x.PERIODO) AS FECHA,
                   x.PERIODO, x.ID_SOCIEDAD, TIPO_MOVIMIENTO, CONCEPTO,
                   ARRAY_CONSTRUCT('Banco Popular', 'Banco Industrial', 'Banco Nacional', 'Banco del Caribe')[MOD(ABS(HASH(CONCEPTO, k)), 4)]::STRING AS BANCO,
                   ROUND(monto * (0.9 + MOD(ABS(HASH(x.PERIODO, CONCEPTO, k, 'm')), 20) / 100), 2) AS USD_MONTO,
                   s.MONEDA,
                   ROUND(monto * (0.9 + MOD(ABS(HASH(x.PERIODO, CONCEPTO, k, 'm')), 20) / 100) * s.TIPO_CAMBIO_USD, 2) AS MONTO_ML
            FROM x JOIN {fq}.DIM_SOCIEDAD s ON s.ID_SOCIEDAD = x.ID_SOCIEDAD""",
    ]

MOTIVOS = [
    ("Quiebre de stock", "el cliente no tiene el producto o se le acabó; pide reposición o el camión no llegó"),
    ("Precio de la competencia", "la competencia ofrece un precio más bajo o una promoción que le quita ventas a ISM"),
    ("Crédito y cobranza", "temas de pago, crédito, facturas vencidas o condiciones de pago del cliente"),
    ("Exhibición y material POP", "nevera, exhibidor, espacio en góndola, material de punto de venta o visibilidad de marca"),
    ("Calidad o producto dañado", "producto vencido, dañado, golpeado, sin gas o con problemas de calidad"),
    ("Pedido adicional o felicitación", "el cliente está contento, aumenta su pedido o pide nuevos productos"),
]

# Templates per motive. [[P]] product, [[C]] competitor brand, [[N]] price, [[D]] days. Index 0-2 Spanish, 3 French (Haiti).
TEMPLATES = {
    "Quiebre de stock": [
        "Visité al cliente y no tenía [[P]] desde hace [[D]] días. Dice que el camión no pasó el martes y está perdiendo ventas, pide reposición urgente.",
        "Cliente reporta que se le agotó [[P]]. Los clientes preguntan y se van a comprar a otro lado. Necesita pedido para esta semana.",
        "Sin inventario de [[P]] en la nevera, solo quedan otros sabores. El dueño dice que la última entrega vino incompleta.",
        "Le client n'a plus de [[P]] depuis [[D]] jours. Le camion n'est pas passé et il perd des ventes, il demande un réapprovisionnement urgent.",
    ],
    "Precio de la competencia": [
        "La competencia ([[C]]) está vendiendo a [[N]] y el cliente dice que [[P]] le sale más caro. Está moviendo menos cajas nuestras.",
        "[[C]] trajo promoción 2x1 y le dejó una nevera nueva al cliente. Pide que mejoremos el precio de [[P]].",
        "El cliente compara [[P]] con [[C]], que está a [[N]]. Dice que si no bajamos el precio va a reducir el pedido.",
        "La concurrence ([[C]]) vend à [[N]] et le client trouve que [[P]] est plus cher. Il vend moins de nos caisses.",
    ],
    "Crédito y cobranza": [
        "El cliente tiene dos facturas vencidas y pide ampliar el crédito a 30 días para seguir comprando [[P]].",
        "No pudo pagar la factura de la semana pasada; pide un plan de pago antes del próximo pedido.",
        "Cliente molesto porque le bloquearon el crédito. Quiere hablar con el supervisor para liberar el pedido de [[P]].",
        "Le client a deux factures en retard et demande un crédit de 30 jours pour continuer à acheter [[P]].",
    ],
    "Exhibición y material POP": [
        "La nevera de ISM está dañada y no enfría bien; el cliente puso [[P]] fuera de la nevera. Pide técnico.",
        "Perdimos el espacio en la góndola principal; ahora [[C]] tiene la cabecera. Solicitar exhibidor nuevo.",
        "El cliente pide material POP y afiches para [[P]]; el letrero actual está despintado.",
        "Le réfrigérateur ISM est en panne et le client a mis [[P]] hors du frigo. Il demande un technicien.",
    ],
    "Calidad o producto dañado": [
        "Llegaron 3 cajas de [[P]] con botellas golpeadas y dos sin gas. El cliente pide nota de crédito.",
        "Cliente encontró [[P]] con fecha vencida en la última entrega. Pide cambio del producto.",
        "Reclamo de calidad: [[P]] con sabor raro según varios consumidores. Retiré una muestra.",
        "Trois caisses de [[P]] sont arrivées avec des bouteilles abîmées. Le client demande un avoir.",
    ],
    "Pedido adicional o felicitación": [
        "Excelente visita: el cliente aumentó su pedido de [[P]] por el fin de semana largo y quiere probar nuevos sabores.",
        "El cliente está muy contento con la entrega puntual y pidió [[D]] cajas más de [[P]].",
        "Buena rotación de [[P]]; el cliente pide agregar el formato familiar a su pedido fijo.",
        "Très bonne visite : le client a augmenté sa commande de [[P]] pour le week-end et veut essayer de nouvelles saveurs.",
    ],
}

COMPETIDORES = ["Cola Brava", "Refresco Sol", "Volt Max", "Agua Pura Andes", "Néctar Dorado", "Energy Shock"]


def _arr(xs):
    return "ARRAY_CONSTRUCT(" + ", ".join("'" + x.replace("'", "''") + "'" for x in xs) + ")"


def notes_statements(fq: str, n: int = 1200, table: str = "NOTAS_VISITA", offset: int = 0, recent_days: int = 90):
    """Builds the sales-rep visit notes table. MOTIVO_REAL is the label a supervisor assigned (ground truth)."""
    motive_names = [m for m, _ in MOTIVOS]
    tmpl_es = ", ".join(_arr(TEMPLATES[m][:3]) for m in motive_names)
    tmpl_fr = _arr([TEMPLATES[m][3] for m in motive_names])
    create = (f"CREATE OR REPLACE TABLE {fq}.{table}" if offset == 0
              else f"INSERT INTO {fq}.{table}")
    comment = " COMMENT = 'Notas de visita de la fuerza de ventas (texto libre). MOTIVO_REAL = clasificación manual del supervisor. Datos sintéticos.'" if offset == 0 else ""
    return [f"""{create}{comment}{' AS' if offset == 0 else ''}
        WITH g AS (SELECT SEQ4() + 1 + {offset} AS n FROM TABLE(GENERATOR(ROWCOUNT => {n}))),
        cli AS (SELECT ID_CUENTA, ID_SOCIEDAD, CANAL_CORPORATIVO, ROW_NUMBER() OVER (ORDER BY ID_CUENTA) - 1 AS rn, COUNT(*) OVER () AS cnt FROM {fq}.DIM_CLIENTE),
        prd AS (SELECT ID_SKU_PRODUCTO, PRODUCTO, ROW_NUMBER() OVER (ORDER BY ID_SKU_PRODUCTO) - 1 AS rn, COUNT(*) OVER () AS cnt FROM {fq}.DIM_PRODUCTO),
        b AS (
          SELECT g.n,
                 MOD(ABS(HASH(g.n, 'm')), 100) AS hm,
                 MOD(ABS(HASH(g.n, 't')), 3) AS ht,
                 MOD(ABS(HASH(g.n, 'c')), 6) AS hcomp,
                 MOD(ABS(HASH(g.n, 'cli')), (SELECT MAX(cnt) FROM cli)) AS hcli,
                 MOD(ABS(HASH(g.n, 'p')), (SELECT MAX(cnt) FROM prd)) AS hp,
                 MOD(ABS(HASH(g.n, 'd')), {recent_days}) AS hd,
                 MOD(ABS(HASH(g.n, 'seed')), 100) AS hseed
          FROM g),
        x AS (
          SELECT b.*, c.ID_CUENTA, c.ID_SOCIEDAD, c.CANAL_CORPORATIVO,
                 -- seeded story: recent Rayo 473ml stock-outs in ISM Dominicana
                 IFF(hseed < 9 AND c.ID_SOCIEDAD = 1000, 'PT3002', p.ID_SKU_PRODUCTO) AS ID_SKU_PRODUCTO,
                 IFF(hseed < 9 AND c.ID_SOCIEDAD = 1000, 0, CASE WHEN hm < 26 THEN 0 WHEN hm < 46 THEN 1 WHEN hm < 58 THEN 2 WHEN hm < 72 THEN 3 WHEN hm < 82 THEN 4 ELSE 5 END) AS mi,
                 IFF(hseed < 9 AND c.ID_SOCIEDAD = 1000, MOD(hd, 20), hd) AS dias_atras
          FROM b JOIN cli c ON c.rn = b.hcli JOIN prd p ON p.rn = b.hp)
        SELECT 'N' || LPAD(n, 6, '0') AS ID_NOTA,
               DATEADD(day, -dias_atras, CURRENT_DATE()) AS FECHA_VISITA,
               x.ID_SOCIEDAD, x.ID_CUENTA, x.ID_SKU_PRODUCTO, x.CANAL_CORPORATIVO,
               'VEND-' || x.ID_SOCIEDAD || '-' || LPAD(MOD(ABS(HASH(x.ID_CUENTA, 'v')), 25) + 1, 2, '0') AS ID_VENDEDOR,
               IFF(x.ID_SOCIEDAD = 3000, 'fr', 'es') AS IDIOMA_ORIGINAL,
               REPLACE(REPLACE(REPLACE(REPLACE(
                 IFF(x.ID_SOCIEDAD = 3000, {tmpl_fr}[mi], ARRAY_CONSTRUCT({tmpl_es})[mi][ht])::STRING,
                 '[[P]]', p.PRODUCTO),
                 '[[C]]', {_arr(COMPETIDORES)}[hcomp]::STRING),
                 '[[N]]', CASE x.ID_SOCIEDAD WHEN 1000 THEN 'RD$ ' WHEN 2000 THEN 'Q ' WHEN 3000 THEN 'HTG ' WHEN 4000 THEN 'S/ ' ELSE 'CLP ' END
                          || TO_VARCHAR(CASE x.ID_SOCIEDAD WHEN 1000 THEN 35 WHEN 2000 THEN 5 WHEN 3000 THEN 75 WHEN 4000 THEN 2 ELSE 650 END + MOD(hcomp * 7, 9))),
                 '[[D]]', TO_VARCHAR(3 + MOD(hd, 9))) AS NOTA,
               {_arr(motive_names)}[mi]::STRING AS MOTIVO_REAL
        FROM x JOIN prd p ON p.ID_SKU_PRODUCTO = x.ID_SKU_PRODUCTO"""]

for stmt in load_statements(fq) + notes_statements(fq, n=1200):
    session.sql(stmt).collect()

for t in ["DIM_SOCIEDAD", "DIM_PRODUCTO", "DIM_CLIENTE", "FACT_DETALLE_VENTAS",
          "ESTADO_RESULTADOS_MENSUAL", "INVENTARIOS_MB52", "FLUJO_EFECTIVO", "NOTAS_VISITA"]:
    n = session.sql(f"SELECT COUNT(*) FROM {fq}.{t}").collect()[0][0]
    print(f"{t:<28} {n:>9,} filas")

In [ ]:
# Revisa sus respuestas de "Su turno" contra sus propios datos. No necesita leer esta celda.
# Las preguntas de texto aceptan cualquier valor empatado en el primer lugar.
def _top(sql_grouped):
    return f"SELECT LISTAGG(k, '|') FROM (SELECT k, RANK() OVER (ORDER BY c DESC) r FROM ({sql_grouped})) WHERE r = 1"

_CHECKS = {
    1: ("number", f"SELECT COUNT(*) FROM {fq}.NOTAS_VISITA WHERE ID_SOCIEDAD = 3000"),
    2: ("text",   _top(f"SELECT MOTIVO_IA k, COUNT(*) c FROM {fq}.NOTAS_ENRIQUECIDAS WHERE ID_SOCIEDAD = 1000 AND FECHA_VISITA >= DATEADD(day, -30, CURRENT_DATE()) GROUP BY 1")),
    3: ("text",   _top(f"SELECT MARCA_COMPETENCIA k, COUNT(*) c FROM {fq}.NOTAS_ENRIQUECIDAS WHERE ID_SOCIEDAD = 2000 AND MARCA_COMPETENCIA IS NOT NULL GROUP BY 1")),
    4: ("number", f"SELECT STATISTICS:numInsertedRows::INT FROM TABLE(INFORMATION_SCHEMA.DYNAMIC_TABLE_REFRESH_HISTORY(NAME_PREFIX => '{fq}.NOTAS_ENRIQUECIDAS')) WHERE REFRESH_ACTION = 'INCREMENTAL' AND STATE = 'SUCCEEDED' ORDER BY REFRESH_START_TIME DESC LIMIT 1"),
    5: ("number", f"SELECT ROUND(SUM(USD_VALOR_NETO)) FROM {fq}.FACT_DETALLE_VENTAS WHERE ID_SOCIEDAD = 4000 AND PERIODO = DATE_TRUNC('month', CURRENT_DATE())"),
    6: ("text",   _top(f"SELECT s.SOCIEDAD k, COUNT(DISTINCT n.ID_CUENTA) c FROM {fq}.NOTAS_ENRIQUECIDAS n JOIN {fq}.DIM_SOCIEDAD s USING (ID_SOCIEDAD) WHERE n.MOTIVO_IA = 'Quiebre de stock' AND n.FECHA_VISITA >= DATEADD(day, -30, CURRENT_DATE()) GROUP BY 1")),
}

def check(n, answer):
    if answer is None:
        print(f"Su turno {n}: escriba su respuesta en la celda y vuelva a ejecutarla.")
        return
    kind, q = _CHECKS[n]
    try:
        expected = session.sql(q).collect()[0][0]
    except Exception:
        expected = None
    if expected is None:
        print(f"Su turno {n}: todavía no existe lo que esta pregunta necesita. Ejecute primero las celdas anteriores.")
        return
    if kind == "number":
        ok = str(answer).strip().replace(",", "").replace("$", "").split(".")[0] == str(int(expected))
    else:
        ok = str(answer).strip().lower() in [e.strip().lower() for e in str(expected).split("|")]
    if ok:
        print(f"Su turno {n}: ¡correcto!")
    else:
        print(f"Su turno {n}: todavía no. Pregúntele a Cortex Code (panel derecho) y vuelva a intentarlo.")

## 2 · Explorar con Cortex Code (10 min)

💬 **Prompt para Cortex Code:**
> *Usando las tablas de {{fq}}, muéstrame la venta neta mensual en USD de Rayo Energy Lata 473ml en ISM Dominicana en los últimos 6 meses, con la variación contra el mes anterior.*

Referencia:

In [ ]:
%%sql -r caida_rayo
SELECT v.PERIODO,
       ROUND(SUM(v.USD_VALOR_NETO)) AS VENTA_NETA_USD,
       ROUND((SUM(v.USD_VALOR_NETO) / LAG(SUM(v.USD_VALOR_NETO)) OVER (ORDER BY v.PERIODO) - 1) * 100, 1) AS VARIACION_PCT
FROM {{fq}}.FACT_DETALLE_VENTAS v
JOIN {{fq}}.DIM_PRODUCTO p USING (ID_SKU_PRODUCTO)
WHERE p.PRODUCTO = 'Rayo Energy Lata 473ml' AND v.ID_SOCIEDAD = 1000
  AND v.PERIODO >= DATEADD(month, -5, DATE_TRUNC('month', CURRENT_DATE()))
GROUP BY 1 ORDER BY 1

In [ ]:
# Vea la caída: venta neta mensual de Rayo Energy Lata 473ml en ISM Dominicana
import altair as alt
import pandas as pd
alt.renderers.enable("mimetype")

_df = caida_rayo.copy() if isinstance(caida_rayo, pd.DataFrame) else caida_rayo.to_pandas()
_df["COLOR"] = _df["VARIACION_PCT"].fillna(0).apply(lambda v: "Caída fuerte" if v < -10 else "Normal")

alt.Chart(_df).mark_bar(cornerRadiusTopLeft=4, cornerRadiusTopRight=4).encode(
    x=alt.X("PERIODO:T", timeUnit="yearmonth", title="Mes"),
    y=alt.Y("VENTA_NETA_USD:Q", title="Venta neta (USD)"),
    color=alt.Color("COLOR:N", scale=alt.Scale(domain=["Normal", "Caída fuerte"], range=["#29B5E8", "#E8463A"]), title=None),
    tooltip=[alt.Tooltip("PERIODO:T", timeUnit="yearmonth", title="Mes"),
             alt.Tooltip("VENTA_NETA_USD:Q", format=",.0f", title="Venta neta"),
             alt.Tooltip("VARIACION_PCT:Q", title="Variación %")],
).properties(title="Rayo Energy Lata 473ml · ISM Dominicana", width=600, height=300)

La caída es clara. Ahora veamos qué dicen los vendedores.

💬 **Prompt:** *Muéstrame 10 notas de visita recientes de ISM Dominicana sobre Rayo Energy Lata 473ml y otras 5 notas de ISM Haití.*

In [ ]:
%%sql -r notas_muestra
(SELECT n.FECHA_VISITA, s.SOCIEDAD, n.IDIOMA_ORIGINAL, n.NOTA
 FROM {{fq}}.NOTAS_VISITA n JOIN {{fq}}.DIM_SOCIEDAD s USING (ID_SOCIEDAD)
 WHERE n.ID_SOCIEDAD = 1000 AND n.ID_SKU_PRODUCTO = 'PT3002' ORDER BY n.FECHA_VISITA DESC LIMIT 10)
UNION ALL
(SELECT n.FECHA_VISITA, s.SOCIEDAD, n.IDIOMA_ORIGINAL, n.NOTA
 FROM {{fq}}.NOTAS_VISITA n JOIN {{fq}}.DIM_SOCIEDAD s USING (ID_SOCIEDAD)
 WHERE n.ID_SOCIEDAD = 3000 ORDER BY n.FECHA_VISITA DESC LIMIT 5)

Las notas mezclan motivos (quiebres, precios de la competencia, neveras dañadas…), vienen en dos idiomas y nadie las clasifica. La columna `MOTIVO_REAL` es la clasificación manual que hizo un supervisor en una muestra: la usaremos **solo para medir a la IA**, nunca como entrada.

### ✏️ Su turno 1
¿Cuántas notas de visita hay de **ISM Haití**? Pregúntele a Cortex Code y escriba el número.

In [ ]:
respuesta_1 = None   # escriba un número, por ejemplo 150
check(1, respuesta_1)

## 3 · Prototipo de IA sobre 10 notas (10 min)

Antes de procesar miles de filas, **pruebe con pocas**. Las AI Functions de Snowflake son funciones SQL: corren dentro de Snowflake, sobre sus datos y con sus permisos.

| Función | Qué hace aquí |
|---|---|
| `AI_TRANSLATE` | Traduce las notas de Haití al español |
| `AI_CLASSIFY` | Asigna uno de 6 motivos de negocio |
| `AI_SENTIMENT` | Tono del cliente: positivo, negativo, neutral |
| `AI_EXTRACT` | Saca la marca y el precio de la competencia |
| `AI_FILTER` | ¿Requiere acción en menos de 48 horas? (sí/no) |

💬 **Prompt:** *Para 10 notas de NOTAS_VISITA, tradúcelas al español si no lo están, clasifica el motivo en estas 6 categorías (Quiebre de stock, Precio de la competencia, Crédito y cobranza, Exhibición y material POP, Calidad o producto dañado, Pedido adicional o felicitación), saca el sentimiento y extrae la marca de la competencia.*

> 💰 **Costo:** las AI Functions se cobran por tokens procesados. Revise las tarifas vigentes en la [documentación de costos de AI Functions](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-cost). Por eso se prueba con 10 filas.

In [ ]:
%%sql -r prototipo_ia
WITH muestra AS (
    SELECT * FROM {{fq}}.NOTAS_VISITA WHERE ID_SOCIEDAD IN (1000, 3000) ORDER BY ID_NOTA LIMIT 10
)
SELECT
    IDIOMA_ORIGINAL,
    NOTA,
    IFF(IDIOMA_ORIGINAL = 'es', NOTA, AI_TRANSLATE(NOTA, IDIOMA_ORIGINAL, 'es')) AS NOTA_ES,
    AI_CLASSIFY(NOTA_ES, [
            {'label': 'Quiebre de stock', 'description': 'el cliente no tiene el producto o se le acabó; pide reposición o el camión no llegó'},
            {'label': 'Precio de la competencia', 'description': 'la competencia ofrece un precio más bajo o una promoción que le quita ventas a ISM'},
            {'label': 'Crédito y cobranza', 'description': 'temas de pago, crédito, facturas vencidas o condiciones de pago del cliente'},
            {'label': 'Exhibición y material POP', 'description': 'nevera, exhibidor, espacio en góndola, material de punto de venta o visibilidad de marca'},
            {'label': 'Calidad o producto dañado', 'description': 'producto vencido, dañado, golpeado, sin gas o con problemas de calidad'},
            {'label': 'Pedido adicional o felicitación', 'description': 'el cliente está contento, aumenta su pedido o pide nuevos productos'}
        ]):labels[0]::STRING AS MOTIVO_IA,
    AI_SENTIMENT(NOTA_ES):categories[0]:sentiment::STRING AS SENTIMIENTO,
    AI_EXTRACT(text => NOTA_ES, responseFormat => {'marca_competencia': 'Marca de la competencia mencionada, o vacío si no hay'}):response:marca_competencia::STRING AS MARCA_COMPETENCIA,
    MOTIVO_REAL
FROM muestra

Compare `MOTIVO_IA` con `MOTIVO_REAL`. Pida a Cortex Code: *"explícame qué hace cada función de IA en esta consulta y qué devuelve AI_CLASSIFY antes de `:labels[0]`"*.

## 4 · Pipeline de IA con una Dynamic Table incremental (8 min)

El prototipo funciona. Ahora llévelo a **todas** las notas con una **Dynamic Table**: una tabla que se mantiene sola a partir de una consulta.

Con `REFRESH_MODE = INCREMENTAL`, cuando lleguen notas nuevas **la IA procesa solo las nuevas**, no las 1,200 otra vez. En IA eso es la diferencia entre pagar una vez o pagar todos los días.

💬 **Prompt:** *Convierte el prototipo en una Dynamic Table incremental llamada NOTAS_ENRIQUECIDAS sobre todas las notas, con target lag de 1 día, que además marque si la nota requiere acción urgente con AI_FILTER y extraiga marca y precio de la competencia.*

Referencia (tarda 2–4 minutos, porque procesa 1,200 notas con 5 funciones de IA):

In [ ]:
%%sql -r crear_notas_enriquecidas_v1
CREATE OR REPLACE DYNAMIC TABLE {{fq}}.NOTAS_ENRIQUECIDAS
    TARGET_LAG = '1 day'
    WAREHOUSE = {{wh}}
    REFRESH_MODE = INCREMENTAL
    COMMENT = 'Notas de visita enriquecidas con IA: traducción, motivo, sentimiento, datos extraídos y urgencia'
AS
SELECT
    n.*,
    IFF(IDIOMA_ORIGINAL = 'es', NOTA, AI_TRANSLATE(NOTA, IDIOMA_ORIGINAL, 'es')) AS NOTA_ES,
    AI_CLASSIFY(
        NOTA_ES,
        [
            {'label': 'Quiebre de stock', 'description': 'el cliente no tiene el producto o se le acabó; pide reposición o el camión no llegó'},
            {'label': 'Precio de la competencia', 'description': 'la competencia ofrece un precio más bajo o una promoción que le quita ventas a ISM'},
            {'label': 'Crédito y cobranza', 'description': 'temas de pago, crédito, facturas vencidas o condiciones de pago del cliente'},
            {'label': 'Exhibición y material POP', 'description': 'nevera, exhibidor, espacio en góndola, material de punto de venta o visibilidad de marca'},
            {'label': 'Calidad o producto dañado', 'description': 'producto vencido, dañado, golpeado, sin gas o con problemas de calidad'},
            {'label': 'Pedido adicional o felicitación', 'description': 'el cliente está contento, aumenta su pedido o pide nuevos productos'}
        ],
        {'task_description': 'Clasifica el motivo principal de la nota que un vendedor de bebidas escribió tras visitar a un cliente'}
    ):labels[0]::STRING AS MOTIVO_IA,
    AI_SENTIMENT(NOTA_ES):categories[0]:sentiment::STRING AS SENTIMIENTO,
    AI_FILTER(PROMPT('Esta nota de visita requiere acción en menos de 48 horas (cliente sin producto, reclamo de calidad o riesgo de perder al cliente): {0}', NOTA_ES)) AS REQUIERE_ACCION_URGENTE,
    AI_EXTRACT(
        text => NOTA_ES,
        responseFormat => {'marca_competencia': 'Marca de la competencia mencionada, o vacío si no hay',
                            'precio_competencia': 'Precio de la competencia mencionado, con moneda, o vacío si no hay'}
    ) AS EXTRAIDO,
    NULLIF(EXTRAIDO:response:marca_competencia::STRING, '') AS MARCA_COMPETENCIA,
    NULLIF(EXTRAIDO:response:precio_competencia::STRING, '') AS PRECIO_COMPETENCIA
FROM {{fq}}.NOTAS_VISITA n

In [ ]:
%%sql -r muestra_enriquecida
SELECT IDIOMA_ORIGINAL, NOTA_ES, MOTIVO_IA, SENTIMIENTO, REQUIERE_ACCION_URGENTE, MARCA_COMPETENCIA, PRECIO_COMPETENCIA
FROM {{fq}}.NOTAS_ENRIQUECIDAS ORDER BY ID_NOTA LIMIT 15

In [ ]:
%%sql -r motivos_por_pais
-- Mapa de calor: qué le preocupa a los clientes en cada país (según la IA)
SELECT s.SOCIEDAD, e.MOTIVO_IA, COUNT(*) AS NOTAS
FROM {{fq}}.NOTAS_ENRIQUECIDAS e JOIN {{fq}}.DIM_SOCIEDAD s USING (ID_SOCIEDAD)
GROUP BY 1, 2

In [ ]:
_df = motivos_por_pais.copy() if isinstance(motivos_por_pais, pd.DataFrame) else motivos_por_pais.to_pandas()

base = alt.Chart(_df).encode(
    x=alt.X("MOTIVO_IA:N", title=None, axis=alt.Axis(labelAngle=-30)),
    y=alt.Y("SOCIEDAD:N", title=None),
)
(base.mark_rect().encode(color=alt.Color("NOTAS:Q", scale=alt.Scale(scheme="blues"), title="Notas"),
                         tooltip=["SOCIEDAD", "MOTIVO_IA", "NOTAS"])
 + base.mark_text(fontSize=12).encode(text="NOTAS:Q",
         color=alt.condition(alt.datum.NOTAS > 60, alt.value("white"), alt.value("black")))
).properties(title="Motivos de visita por país (clasificados con AI_CLASSIFY)", width=650, height=230)

### ✏️ Su turno 2
Según la IA, ¿cuál es el **motivo más frecuente** en las notas de **ISM Dominicana de los últimos 30 días**? Escríbalo tal como aparece en `MOTIVO_IA`.

In [ ]:
respuesta_2 = None   # escriba el motivo entre comillas, por ejemplo "Crédito y cobranza"
check(2, respuesta_2)

## 5 · Validar la IA antes de confiar en ella (15 min)

La clasificación se ve bien. ¿Pero el resto? **Una IA que se equivoca en silencio es peor que no tener IA.** Este reporte mide tres errores que sí podemos comprobar:

1. **¿La traducción cambió el nombre de nuestro producto?** Una nota en francés que menciona "Kola Isla" debe seguir diciendo "Kola Isla".
2. **¿La IA dijo que una marca nuestra es "competencia"?**
3. **¿La IA devolvió el texto "None" en lugar de dejar el campo vacío?** Eso ensucia cualquier conteo.

In [ ]:
%%sql -r reporte_calidad_v1
SELECT
    COUNT(*) AS NOTAS,
    ROUND(AVG(IFF(e.MOTIVO_IA = e.MOTIVO_REAL, 1, 0)) * 100, 1) AS PCT_MOTIVO_CORRECTO,
    COUNT_IF(e.IDIOMA_ORIGINAL <> 'es' AND NOT CONTAINS(e.NOTA_ES, p.PRODUCTO)) AS TRADUCCION_CAMBIO_EL_PRODUCTO,
    COUNT_IF(e.MARCA_COMPETENCIA ILIKE ANY ('%Kola Isla%', '%Fresquita%', '%Agua Montaña%', '%Rayo%', '%Frutalia%', '%Hidra+%', '%Té Brisa%')) AS MARCA_PROPIA_COMO_COMPETENCIA,
    COUNT_IF(LOWER(TRIM(e.MARCA_COMPETENCIA)) IN ('none', 'null', 'n/a', 'ninguna', 'ninguno')) AS TEXTO_NONE_EN_VEZ_DE_VACIO
FROM {{fq}}.NOTAS_ENRIQUECIDAS e
JOIN {{fq}}.DIM_PRODUCTO p ON p.ID_SKU_PRODUCTO = e.ID_SKU_PRODUCTO

Ahí están los errores. Ninguno se ve a simple vista, y todos distorsionan los tableros: "la competencia más mencionada" incluiría a Rayo, y "None" aparecería como una marca.

**La corrección (v2)** combina dos cosas, y esa es la lección de esta sección:
- **Mejor contexto para la IA:** el prompt de extracción ahora sabe cuáles son las marcas propias de ISM, y `AI_CLASSIFY` lee la nota original en vez de la traducción.
- **Reglas determinísticas como red de seguridad:** se protege el nombre del producto durante la traducción, se descartan las marcas propias y "None" se convierte en `NULL`.

💬 **Prompt:** *Corrige NOTAS_ENRIQUECIDAS: (1) protege el nombre del producto durante AI_TRANSLATE, (2) en AI_EXTRACT aclara que las marcas de ISM no son competencia y descártalas si aparecen, (3) convierte 'None' en NULL. Mantén REFRESH_MODE = INCREMENTAL.*

In [ ]:
%%sql -r crear_notas_enriquecidas_v2
CREATE OR REPLACE DYNAMIC TABLE {{fq}}.NOTAS_ENRIQUECIDAS
    TARGET_LAG = '1 day'
    WAREHOUSE = {{wh}}
    REFRESH_MODE = INCREMENTAL
    COMMENT = 'Notas de visita enriquecidas con IA (v2): traducción, motivo, sentimiento, datos extraídos y urgencia'
AS
SELECT
    n.*,
    IFF(IDIOMA_ORIGINAL = 'es', NOTA,
        REPLACE(AI_TRANSLATE(REPLACE(NOTA, p.PRODUCTO, 'XSKUX'), IDIOMA_ORIGINAL, 'es'), 'XSKUX', p.PRODUCTO)) AS NOTA_ES,
    AI_CLASSIFY(
        NOTA,
        [
            {'label': 'Quiebre de stock', 'description': 'el cliente no tiene el producto o se le acabó; pide reposición o el camión no llegó'},
            {'label': 'Precio de la competencia', 'description': 'la competencia ofrece un precio más bajo o una promoción que le quita ventas a ISM'},
            {'label': 'Crédito y cobranza', 'description': 'temas de pago, crédito, facturas vencidas o condiciones de pago del cliente'},
            {'label': 'Exhibición y material POP', 'description': 'nevera, exhibidor, espacio en góndola, material de punto de venta o visibilidad de marca'},
            {'label': 'Calidad o producto dañado', 'description': 'producto vencido, dañado, golpeado, sin gas o con problemas de calidad'},
            {'label': 'Pedido adicional o felicitación', 'description': 'el cliente está contento, aumenta su pedido o pide nuevos productos'}
        ],
        {'task_description': 'Clasifica el motivo principal de la nota que un vendedor de bebidas escribió tras visitar a un cliente'}
    ):labels[0]::STRING AS MOTIVO_IA,
    AI_SENTIMENT(NOTA_ES):categories[0]:sentiment::STRING AS SENTIMIENTO,
    AI_FILTER(PROMPT('Esta nota de visita requiere acción en menos de 48 horas (cliente sin producto, reclamo de calidad o riesgo de perder al cliente): {0}', NOTA_ES)) AS REQUIERE_ACCION_URGENTE,
    AI_EXTRACT(
        text => NOTA_ES,
        responseFormat => {'marca_competencia': 'Marca de la COMPETENCIA mencionada. Las marcas de ISM (Kola Isla, Fresquita, Agua Montaña, Rayo, Frutalia, Hidra+, Té Brisa) NO son competencia: si solo aparecen marcas de ISM, deja vacío',
                            'precio_competencia': 'Precio de la competencia mencionado, con moneda, o vacío si no hay'}
    ) AS EXTRAIDO,
    IFF(EXTRAIDO:response:marca_competencia::STRING ILIKE ANY ('%Kola Isla%', '%Fresquita%', '%Agua Montaña%', '%Rayo%', '%Frutalia%', '%Hidra+%', '%Té Brisa%')
        OR LOWER(TRIM(EXTRAIDO:response:marca_competencia::STRING)) IN ('', 'none', 'null', 'n/a', 'ninguna', 'ninguno'),
        NULL, EXTRAIDO:response:marca_competencia::STRING) AS MARCA_COMPETENCIA,
    IFF(LOWER(TRIM(EXTRAIDO:response:precio_competencia::STRING)) IN ('', 'none', 'null', 'n/a', 'ninguna', 'ninguno'),
        NULL, EXTRAIDO:response:precio_competencia::STRING) AS PRECIO_COMPETENCIA
FROM {{fq}}.NOTAS_VISITA n
JOIN {{fq}}.DIM_PRODUCTO p ON p.ID_SKU_PRODUCTO = n.ID_SKU_PRODUCTO

In [ ]:
%%sql -r reporte_calidad_v2
SELECT
    COUNT(*) AS NOTAS,
    ROUND(AVG(IFF(e.MOTIVO_IA = e.MOTIVO_REAL, 1, 0)) * 100, 1) AS PCT_MOTIVO_CORRECTO,
    COUNT_IF(e.IDIOMA_ORIGINAL <> 'es' AND NOT CONTAINS(e.NOTA_ES, p.PRODUCTO)) AS TRADUCCION_CAMBIO_EL_PRODUCTO,
    COUNT_IF(e.MARCA_COMPETENCIA ILIKE ANY ('%Kola Isla%', '%Fresquita%', '%Agua Montaña%', '%Rayo%', '%Frutalia%', '%Hidra+%', '%Té Brisa%')) AS MARCA_PROPIA_COMO_COMPETENCIA,
    COUNT_IF(LOWER(TRIM(e.MARCA_COMPETENCIA)) IN ('none', 'null', 'n/a', 'ninguna', 'ninguno')) AS TEXTO_NONE_EN_VEZ_DE_VACIO
FROM {{fq}}.NOTAS_ENRIQUECIDAS e
JOIN {{fq}}.DIM_PRODUCTO p ON p.ID_SKU_PRODUCTO = e.ID_SKU_PRODUCTO

In [ ]:
# Antes y después: errores de la IA en v1 vs v2
_to_pd = lambda d: d.copy() if isinstance(d, pd.DataFrame) else d.to_pandas()
_err = ["TRADUCCION_CAMBIO_EL_PRODUCTO", "MARCA_PROPIA_COMO_COMPETENCIA", "TEXTO_NONE_EN_VEZ_DE_VACIO"]
_cmp = pd.concat([
    _to_pd(reporte_calidad_v1)[_err].melt(var_name="ERROR", value_name="CASOS").assign(VERSION="v1"),
    _to_pd(reporte_calidad_v2)[_err].melt(var_name="ERROR", value_name="CASOS").assign(VERSION="v2"),
])
_cmp["ERROR"] = _cmp["ERROR"].str.replace("_", " ").str.capitalize()

alt.Chart(_cmp).mark_bar().encode(
    x=alt.X("CASOS:Q", title="Notas afectadas"),
    y=alt.Y("VERSION:N", title=None),
    color=alt.Color("VERSION:N", scale=alt.Scale(range=["#E8463A", "#2E9E5B"]), legend=None),
    row=alt.Row("ERROR:N", title=None, header=alt.Header(labelAngle=0, labelAlign="left")),
    tooltip=["VERSION", "ERROR", "CASOS"],
).properties(width=500, height=40, title="Errores de la IA: v1 vs v2")

**Errores en cero.** Confirme que la Dynamic Table sigue en modo incremental: si el modo fuera `FULL`, cada refresco volvería a pagar la IA de todas las filas.

In [ ]:
%%sql -r modo_refresco
SHOW DYNAMIC TABLES IN SCHEMA {{fq}}
->> SELECT "name" AS DYNAMIC_TABLE, "refresh_mode" AS MODO, "refresh_mode_reason" AS RAZON, "rows" AS FILAS FROM $1

### ✏️ Su turno 3
Con la v2, ¿qué **marca de la competencia** mencionan más los vendedores de **ISM Guatemala**?

In [ ]:
respuesta_3 = None   # por ejemplo "Cola Brava"
check(3, respuesta_3)

## 6 · Llega un lote nuevo: la IA procesa solo lo nuevo (7 min)

Mañana llegan 100 notas nuevas de los vendedores. Simulémoslo:

In [ ]:
for stmt in notes_statements(fq, n=100, offset=1200, recent_days=3):
    session.sql(stmt).collect()
print("Notas en NOTAS_VISITA:", session.sql(f"SELECT COUNT(*) FROM {fq}.NOTAS_VISITA").collect()[0][0])

En producción, el refresco ocurre solo según el `TARGET_LAG`. Aquí lo forzamos para verlo ya:

In [ ]:
%%sql -r refrescar
ALTER DYNAMIC TABLE {{fq}}.NOTAS_ENRIQUECIDAS REFRESH

In [ ]:
%%sql -r historial_refrescos
SELECT REFRESH_START_TIME, REFRESH_ACTION, STATE,
       STATISTICS:numInsertedRows::INT AS FILAS_PROCESADAS_CON_IA
FROM TABLE(INFORMATION_SCHEMA.DYNAMIC_TABLE_REFRESH_HISTORY(NAME_PREFIX => '{{fq}}.NOTAS_ENRIQUECIDAS'))
ORDER BY REFRESH_START_TIME DESC
LIMIT 5

El último refresco fue `INCREMENTAL` y procesó solo las notas nuevas.

💬 **Prompt:** *¿Cuántas llamadas a funciones de IA nos ahorramos con el refresco incremental frente a reprocesar todo?*

### ✏️ Su turno 4
¿Cuántas filas procesó con IA el **último refresco incremental**?

In [ ]:
respuesta_4 = None   # un número
check(4, respuesta_4)

## 7 · El modelo semántico: la cifra que no cuadra con Finanzas (12 min)

Para que cualquier persona pueda preguntar en español, la IA necesita un **modelo semántico**: qué tablas hay, cómo se unen y qué significa cada métrica.

💬 **Prompt:** *Crea una semantic view SV_COMERCIAL_ISM con ventas, productos, clientes, sociedades y NOTAS_ENRIQUECIDAS, con una métrica de ventas y métricas de notas (número de notas, urgentes y clientes con quiebre).*

Esta es la primera versión que suele salir: a "ventas" la mapea a la primera columna que parece ventas, `USD_VALOR_BRUTO`.

In [ ]:
%%sql -r semantic_view_v1
CREATE OR REPLACE SEMANTIC VIEW {{fq}}.SV_COMERCIAL_ISM
  TABLES (
    sociedades AS {{fq}}.DIM_SOCIEDAD PRIMARY KEY (ID_SOCIEDAD)
      WITH SYNONYMS = ('país', 'empresa', 'compañía') COMMENT = 'Sociedades de ISM, una por país',
    productos AS {{fq}}.DIM_PRODUCTO PRIMARY KEY (ID_SKU_PRODUCTO)
      WITH SYNONYMS = ('sku', 'artículo') COMMENT = 'Productos terminados de ISM (marcas propias)',
    clientes AS {{fq}}.DIM_CLIENTE PRIMARY KEY (ID_CUENTA)
      WITH SYNONYMS = ('cliente', 'punto de venta', 'pdv') COMMENT = 'Clientes con canal y ciudad',
    ventas AS {{fq}}.FACT_DETALLE_VENTAS PRIMARY KEY (ID_VENTA)
      COMMENT = 'Ventas mensuales por cliente y SKU (USD)',
    notas AS {{fq}}.NOTAS_ENRIQUECIDAS PRIMARY KEY (ID_NOTA)
      WITH SYNONYMS = ('visitas', 'notas de visita', 'comentarios del vendedor', 'feedback de clientes')
      COMMENT = 'Notas de visita de los vendedores, enriquecidas con IA (motivo, sentimiento, urgencia, competencia)'
  )
  RELATIONSHIPS (
    ventas_sociedad AS ventas (ID_SOCIEDAD) REFERENCES sociedades,
    ventas_producto AS ventas (ID_SKU_PRODUCTO) REFERENCES productos,
    ventas_cliente AS ventas (ID_CUENTA) REFERENCES clientes,
    notas_sociedad AS notas (ID_SOCIEDAD) REFERENCES sociedades,
    notas_producto AS notas (ID_SKU_PRODUCTO) REFERENCES productos,
    notas_cliente AS notas (ID_CUENTA) REFERENCES clientes
  )
  DIMENSIONS (
    sociedades.sociedad AS SOCIEDAD WITH SYNONYMS = ('país', 'empresa') COMMENT = 'ISM Dominicana, ISM Guatemala, ISM Haití, ISM Perú o ISM Chile',
    sociedades.pais AS PAIS COMMENT = 'País de la sociedad',
    productos.producto AS PRODUCTO WITH SYNONYMS = ('sku') COMMENT = 'Nombre del producto',
    productos.marca AS MARCA COMMENT = 'Marca propia de ISM',
    productos.categoria AS CATEGORIA COMMENT = 'Gaseosas, Agua, Energizantes, Jugos y Néctares, Isotónicos, Té',
    clientes.cliente AS NOMBRE_CLIENTE COMMENT = 'Nombre del cliente',
    clientes.canal AS CANAL_CORPORATIVO WITH SYNONYMS = ('canal') COMMENT = 'Tradicional, Moderno, Mayorista u On Premise',
    clientes.ciudad AS CIUDAD COMMENT = 'Ciudad del cliente',
    ventas.mes_venta AS PERIODO WITH SYNONYMS = ('mes') COMMENT = 'Mes de la venta (primer día del mes)',
    ventas.es_mes_actual AS PERIODO = DATE_TRUNC('month', CURRENT_DATE()) WITH SYNONYMS = ('este mes', 'mes actual') COMMENT = 'TRUE para el mes en curso',
    notas.fecha_visita AS FECHA_VISITA COMMENT = 'Fecha de la visita',
    notas.motivo AS MOTIVO_IA WITH SYNONYMS = ('razón', 'causa', 'por qué', 'motivo de la visita') COMMENT = 'Motivo de la nota clasificado con IA: Quiebre de stock, Precio de la competencia, Crédito y cobranza, Exhibición y material POP, Calidad o producto dañado, Pedido adicional o felicitación',
    notas.sentimiento AS SENTIMIENTO COMMENT = 'Sentimiento del cliente según la nota: positive, negative, neutral o mixed',
    notas.requiere_accion_urgente AS REQUIERE_ACCION_URGENTE WITH SYNONYMS = ('urgente', 'crítico') COMMENT = 'TRUE si la nota requiere acción en menos de 48 horas',
    notas.marca_competencia AS MARCA_COMPETENCIA WITH SYNONYMS = ('competidor', 'competencia') COMMENT = 'Marca de la competencia mencionada en la nota',
    notas.precio_competencia AS PRECIO_COMPETENCIA COMMENT = 'Precio de la competencia mencionado, con moneda local',
    notas.nota AS NOTA_ES WITH SYNONYMS = ('comentario', 'texto de la nota') COMMENT = 'Texto de la nota en español',
    notas.es_ultimos_30_dias AS FECHA_VISITA >= DATEADD(day, -30, CURRENT_DATE()) WITH SYNONYMS = ('último mes', 'recientes') COMMENT = 'TRUE para visitas de los últimos 30 días'
  )
  METRICS (
    ventas.ventas AS SUM(USD_VALOR_BRUTO) WITH SYNONYMS = ('venta', 'cuánto vendimos', 'facturación') COMMENT = 'Ventas (USD)',
    ventas.cajas_vendidas AS SUM(CANTIDAD_CAJAS) COMMENT = 'Cajas vendidas',
    notas.numero_notas AS COUNT(ID_NOTA) WITH SYNONYMS = ('cantidad de notas', 'número de visitas', 'reportes') COMMENT = 'Número de notas de visita',
    notas.notas_urgentes AS COUNT_IF(REQUIERE_ACCION_URGENTE) COMMENT = 'Número de notas que requieren acción urgente',
    notas.clientes_con_quiebre AS COUNT(DISTINCT IFF(MOTIVO_IA = 'Quiebre de stock', ID_CUENTA, NULL)) WITH SYNONYMS = ('clientes sin producto') COMMENT = 'Clientes distintos que reportaron quiebre de stock'
  )
  COMMENT = 'Modelo comercial ISM v1'

In [ ]:
%%sql -r ventas_v1_vs_finanzas
SELECT sv.SOCIEDAD,
       ROUND(sv.VENTAS) AS VENTAS_SEGUN_MODELO_V1,
       ROUND(f.VENTA_NETA) AS VENTA_NETA_FINANZAS,
       ROUND(sv.VENTAS - f.VENTA_NETA) AS DIFERENCIA
FROM SEMANTIC_VIEW({{fq}}.SV_COMERCIAL_ISM DIMENSIONS sociedades.sociedad METRICS ventas.ventas WHERE ventas.es_mes_actual) sv
JOIN (SELECT s.SOCIEDAD, SUM(USD_VALOR_NETO) AS VENTA_NETA
      FROM {{fq}}.FACT_DETALLE_VENTAS v JOIN {{fq}}.DIM_SOCIEDAD s USING (ID_SOCIEDAD)
      WHERE PERIODO = DATE_TRUNC('month', CURRENT_DATE()) GROUP BY 1) f USING (SOCIEDAD)
ORDER BY 2 DESC

**Millones de dólares de diferencia.** El modelo v1 reporta la venta **bruta**; Finanzas reporta la venta **neta** (bruta − descuentos − bonificaciones − rechazos). Si el agente usara la v1, Comercial y Finanzas llevarían dos cifras distintas a la misma reunión.

**La corrección no va en el prompt, va en el modelo:** se redefine "ventas" una vez y todos los agentes y tableros quedan bien.

💬 **Prompt:** *En SV_COMERCIAL_ISM, define 'ventas' como la venta neta (USD_VALOR_NETO), deja la venta bruta como una métrica aparte y agrega margen, bonificaciones y la variación contra el mes anterior.*

In [ ]:
%%sql -r semantic_view_v2
CREATE OR REPLACE SEMANTIC VIEW {{fq}}.SV_COMERCIAL_ISM
  TABLES (
    sociedades AS {{fq}}.DIM_SOCIEDAD PRIMARY KEY (ID_SOCIEDAD)
      WITH SYNONYMS = ('país', 'empresa', 'compañía') COMMENT = 'Sociedades de ISM, una por país',
    productos AS {{fq}}.DIM_PRODUCTO PRIMARY KEY (ID_SKU_PRODUCTO)
      WITH SYNONYMS = ('sku', 'artículo') COMMENT = 'Productos terminados de ISM (marcas propias)',
    clientes AS {{fq}}.DIM_CLIENTE PRIMARY KEY (ID_CUENTA)
      WITH SYNONYMS = ('cliente', 'punto de venta', 'pdv') COMMENT = 'Clientes con canal y ciudad',
    ventas AS {{fq}}.FACT_DETALLE_VENTAS PRIMARY KEY (ID_VENTA)
      COMMENT = 'Ventas mensuales por cliente y SKU (USD)',
    notas AS {{fq}}.NOTAS_ENRIQUECIDAS PRIMARY KEY (ID_NOTA)
      WITH SYNONYMS = ('visitas', 'notas de visita', 'comentarios del vendedor', 'feedback de clientes')
      COMMENT = 'Notas de visita de los vendedores, enriquecidas con IA (motivo, sentimiento, urgencia, competencia)'
  )
  RELATIONSHIPS (
    ventas_sociedad AS ventas (ID_SOCIEDAD) REFERENCES sociedades,
    ventas_producto AS ventas (ID_SKU_PRODUCTO) REFERENCES productos,
    ventas_cliente AS ventas (ID_CUENTA) REFERENCES clientes,
    notas_sociedad AS notas (ID_SOCIEDAD) REFERENCES sociedades,
    notas_producto AS notas (ID_SKU_PRODUCTO) REFERENCES productos,
    notas_cliente AS notas (ID_CUENTA) REFERENCES clientes
  )
  DIMENSIONS (
    sociedades.sociedad AS SOCIEDAD WITH SYNONYMS = ('país', 'empresa') COMMENT = 'ISM Dominicana, ISM Guatemala, ISM Haití, ISM Perú o ISM Chile',
    sociedades.pais AS PAIS COMMENT = 'País de la sociedad',
    productos.producto AS PRODUCTO WITH SYNONYMS = ('sku') COMMENT = 'Nombre del producto',
    productos.marca AS MARCA COMMENT = 'Marca propia de ISM',
    productos.categoria AS CATEGORIA COMMENT = 'Gaseosas, Agua, Energizantes, Jugos y Néctares, Isotónicos, Té',
    clientes.cliente AS NOMBRE_CLIENTE COMMENT = 'Nombre del cliente',
    clientes.canal AS CANAL_CORPORATIVO WITH SYNONYMS = ('canal') COMMENT = 'Tradicional, Moderno, Mayorista u On Premise',
    clientes.ciudad AS CIUDAD COMMENT = 'Ciudad del cliente',
    ventas.mes_venta AS PERIODO WITH SYNONYMS = ('mes') COMMENT = 'Mes de la venta (primer día del mes)',
    ventas.es_mes_actual AS PERIODO = DATE_TRUNC('month', CURRENT_DATE()) WITH SYNONYMS = ('este mes', 'mes actual') COMMENT = 'TRUE para el mes en curso',
    notas.fecha_visita AS FECHA_VISITA COMMENT = 'Fecha de la visita',
    notas.motivo AS MOTIVO_IA WITH SYNONYMS = ('razón', 'causa', 'por qué', 'motivo de la visita') COMMENT = 'Motivo de la nota clasificado con IA: Quiebre de stock, Precio de la competencia, Crédito y cobranza, Exhibición y material POP, Calidad o producto dañado, Pedido adicional o felicitación',
    notas.sentimiento AS SENTIMIENTO COMMENT = 'Sentimiento del cliente según la nota: positive, negative, neutral o mixed',
    notas.requiere_accion_urgente AS REQUIERE_ACCION_URGENTE WITH SYNONYMS = ('urgente', 'crítico') COMMENT = 'TRUE si la nota requiere acción en menos de 48 horas',
    notas.marca_competencia AS MARCA_COMPETENCIA WITH SYNONYMS = ('competidor', 'competencia') COMMENT = 'Marca de la competencia mencionada en la nota',
    notas.precio_competencia AS PRECIO_COMPETENCIA COMMENT = 'Precio de la competencia mencionado, con moneda local',
    notas.nota AS NOTA_ES WITH SYNONYMS = ('comentario', 'texto de la nota') COMMENT = 'Texto de la nota en español',
    notas.es_ultimos_30_dias AS FECHA_VISITA >= DATEADD(day, -30, CURRENT_DATE()) WITH SYNONYMS = ('último mes', 'recientes') COMMENT = 'TRUE para visitas de los últimos 30 días'
  )
  METRICS (
    ventas.ventas AS SUM(USD_VALOR_NETO) WITH SYNONYMS = ('venta', 'venta neta', 'cuánto vendimos', 'facturación')
      COMMENT = 'Ventas = VENTA NETA (USD): bruta - descuentos - bonificaciones - rechazos. Es la cifra oficial de Finanzas',
    ventas.venta_bruta AS SUM(USD_VALOR_BRUTO) COMMENT = 'Venta bruta a precio de lista (USD). Usar solo si piden explícitamente la venta bruta',
    ventas.bonificaciones AS SUM(USD_BONIFICACIONES) COMMENT = 'Bonificaciones (USD)',
    ventas.margen_bruto AS SUM(USD_VALOR_NETO - USD_COSTO_VENTA) WITH SYNONYMS = ('margen') COMMENT = 'Venta neta - costo de ventas (USD)',
    ventas.cajas_vendidas AS SUM(CANTIDAD_CAJAS) COMMENT = 'Cajas vendidas',
    ventas.venta_mes_actual AS SUM(IFF(PERIODO = DATE_TRUNC('month', CURRENT_DATE()), USD_VALOR_NETO, 0)) COMMENT = 'Venta neta del mes en curso (USD)',
    ventas.venta_mes_anterior AS SUM(IFF(PERIODO = DATEADD(month, -1, DATE_TRUNC('month', CURRENT_DATE())), USD_VALOR_NETO, 0)) COMMENT = 'Venta neta del mes anterior (USD)',
    ventas.pct_variacion_vs_mes_anterior AS (SUM(IFF(PERIODO = DATE_TRUNC('month', CURRENT_DATE()), USD_VALOR_NETO, 0)) - SUM(IFF(PERIODO = DATEADD(month, -1, DATE_TRUNC('month', CURRENT_DATE())), USD_VALOR_NETO, 0)))
        / NULLIF(SUM(IFF(PERIODO = DATEADD(month, -1, DATE_TRUNC('month', CURRENT_DATE())), USD_VALOR_NETO, 0)), 0)
      WITH SYNONYMS = ('caída', 'variación') COMMENT = 'Variación de la venta neta del mes actual vs el anterior (-0.30 = cayó 30%)',
    notas.numero_notas AS COUNT(ID_NOTA) WITH SYNONYMS = ('cantidad de notas', 'número de visitas', 'reportes') COMMENT = 'Número de notas de visita',
    notas.notas_urgentes AS COUNT_IF(REQUIERE_ACCION_URGENTE) COMMENT = 'Número de notas que requieren acción urgente',
    notas.clientes_con_quiebre AS COUNT(DISTINCT IFF(MOTIVO_IA = 'Quiebre de stock', ID_CUENTA, NULL)) WITH SYNONYMS = ('clientes sin producto') COMMENT = 'Clientes distintos que reportaron quiebre de stock'
  )
  COMMENT = 'Modelo comercial ISM v2: ventas = venta neta (definición de Finanzas) + notas de visita enriquecidas con IA'

In [ ]:
%%sql -r ventas_v2
SELECT * FROM SEMANTIC_VIEW(
    {{fq}}.SV_COMERCIAL_ISM
    DIMENSIONS sociedades.sociedad
    METRICS ventas.ventas, ventas.venta_bruta, ventas.pct_variacion_vs_mes_anterior
    WHERE ventas.es_mes_actual
) ORDER BY ventas DESC

### ✏️ Su turno 5
¿Cuánto vendió **ISM Perú** este mes (venta neta, en USD, sin decimales)?

In [ ]:
respuesta_5 = None   # un número sin decimales, por ejemplo 1234567
check(5, respuesta_5)

## 8 · El agente comercial, publicado en CoWork (10 min)

El **Asistente Comercial ISM** usa el modelo semántico v2 para responder sobre ventas **y** sobre la voz del cliente, para todos los países.

💬 **Prompt:** *Crea un Cortex Agent llamado ASISTENTE_COMERCIAL_ISM que use SV_COMERCIAL_ISM, responda en el idioma del usuario y, para explicar caídas de ventas, combine la variación con las notas de visita.*

In [ ]:
%%sql -r crear_agente
CREATE OR REPLACE AGENT {{fq}}.ASISTENTE_COMERCIAL_ISM
  COMMENT = 'ISM Snow Day: asistente comercial multipaís (ventas + voz del cliente)'
  PROFILE = '{"display_name": "Asistente Comercial ISM"}'
  FROM SPECIFICATION
  $$
  models:
    orchestration: auto

  instructions:
    response: "Eres el asistente comercial de ISM para todos los países. Responde en el idioma del usuario (español por defecto), en tono ejecutivo: la cifra clave primero, luego una tabla corta y una recomendación. Montos en USD con separador de miles. Si la pregunta es de un país, filtra por su sociedad."
    orchestration: "Usa comercial_ism para todo. Ventas significa venta neta. Para explicar POR QUÉ cambian las ventas de un producto o cliente, combina la variación de ventas con las notas de visita (motivo, urgencia y competencia) del mismo producto y sociedad en los últimos 30 días."
    sample_questions:
      - question: "¿Por qué cayeron las ventas de Rayo Energy Lata 473ml en ISM Dominicana?"
      - question: "¿Qué marcas de la competencia mencionan más los vendedores en Guatemala?"
      - question: "¿Cuántos clientes reportaron quiebre de stock en los últimos 30 días, por sociedad?"
      - question: "¿Cuánto vendimos este mes en ISM Perú por canal?"

  tools:
    - tool_spec:
        type: "cortex_analyst_text_to_sql"
        name: "comercial_ism"
        description: "Ventas netas, margen, bonificaciones y cajas por sociedad (país), producto, marca, categoría, canal y cliente; y notas de visita de los vendedores enriquecidas con IA: motivo, sentimiento, urgencia, marca y precio de la competencia."

  tool_resources:
    comercial_ism:
      semantic_view: "{{fq}}.SV_COMERCIAL_ISM"
      execution_environment:
        type: "warehouse"
        warehouse: "{{wh}}"
  $$

In [ ]:
agent_name = "ASISTENTE_COMERCIAL_ISM"
display_name = "Asistente Comercial ISM"

agent_fqn = f"{fq}.{agent_name}"
si_objects = session.sql("SHOW SNOWFLAKE INTELLIGENCES").collect()

if not si_objects:
    print("Esta cuenta no tiene objeto CoWork: los agentes aparecen en CoWork automáticamente.")
else:
    si_name = si_objects[0]["name"]
    try:
        session.sql(f"ALTER SNOWFLAKE INTELLIGENCE {si_name} ADD AGENT {agent_fqn}").collect()
        print(f"Agente {agent_fqn} agregado a CoWork ({si_name}).")
    except Exception as e:
        if "already" in str(e).lower():
            print(f"El agente {agent_fqn} ya estaba en CoWork ({si_name}).")
        else:
            print(f"No se pudo agregar el agente a {si_name}. Verifique que la celda de setup corrió como ACCOUNTADMIN.\n{e}")

print(f"Abra CoWork en https://ai.snowflake.com y busque '{display_name}'.")

Ahora la pregunta del inicio. Puede hacerla en **CoWork** (https://ai.snowflake.com) o desde aquí, porque `DATA_AGENT_RUN` llama al agente con SQL:

In [ ]:
import json

def preguntar(pregunta, agente=f"{fq}.ASISTENTE_COMERCIAL_ISM"):
    """Llama al agente desde SQL con DATA_AGENT_RUN y devuelve el texto de la respuesta."""
    body = json.dumps({"messages": [{"role": "user", "content": [{"type": "text", "text": pregunta}]}]}, ensure_ascii=False)
    body = body.replace("$$", "")
    raw = session.sql(f"SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN('{agente}', $${body}$$)").collect()[0][0]
    content = json.loads(raw).get("content", [])
    return "\n".join(c.get("text", "") for c in content if c.get("type") == "text").strip()

print(preguntar("¿Por qué cayeron las ventas de Rayo Energy Lata 473ml en ISM Dominicana este mes?"))

In [ ]:
# Pregunta libre: cambie el texto y ejecute la celda. Pruebe en español o en francés.
mi_pregunta = "¿Qué 5 clientes de ISM Dominicana tienen más notas urgentes y qué producto les falta?"
print(preguntar(mi_pregunta))

**Ventas + voz del cliente en una sola respuesta:** la caída de Rayo coincide con decenas de notas urgentes de quiebre de stock. No es la competencia, es abastecimiento. Esa es una conversación concreta para Supply mañana temprano.

### ✏️ Su turno 6
Pregúntele al agente (en CoWork o con `preguntar(...)`): ¿qué **sociedad** tiene **más clientes con quiebre de stock** en los últimos 30 días?

In [ ]:
respuesta_6 = None   # por ejemplo "ISM Chile"
check(6, respuesta_6)

## 9 · Evaluación multipaís (8 min)

Antes de entregarle el agente a 5 países, **mídalo**. Esta celda le hace preguntas de distintos países (una en francés) y muestra al lado la respuesta correcta calculada en SQL.

💬 **Prompt:** *Agrega al set de evaluación dos preguntas que mi equipo comercial haría en Guatemala y Chile, con su respuesta esperada en SQL.*

In [ ]:
# Set de evaluación: pregunta -> respuesta esperada calculada en SQL. Revise si el agente acierta.
# Si hay empate en el primer lugar, se aceptan todas las respuestas empatadas (separadas por " o ").
evaluacion = [
    ("¿Cuánto vendimos este mes en ISM Perú?",
     f"SELECT 'USD ' || TO_VARCHAR(ROUND(SUM(USD_VALOR_NETO)), 'FM999,999,999') FROM {fq}.FACT_DETALLE_VENTAS WHERE ID_SOCIEDAD = 4000 AND PERIODO = DATE_TRUNC('month', CURRENT_DATE())"),
    ("¿Qué marca de la competencia mencionan más los vendedores en ISM Guatemala?",
     f"SELECT LISTAGG(k, ' o ') FROM (SELECT MARCA_COMPETENCIA k, RANK() OVER (ORDER BY COUNT(*) DESC) r FROM {fq}.NOTAS_ENRIQUECIDAS WHERE ID_SOCIEDAD = 2000 AND MARCA_COMPETENCIA IS NOT NULL GROUP BY 1) WHERE r = 1"),
    ("¿Cuántos clientes de ISM Dominicana reportaron quiebre de stock en los últimos 30 días?",
     f"SELECT COUNT(DISTINCT ID_CUENTA) FROM {fq}.NOTAS_ENRIQUECIDAS WHERE ID_SOCIEDAD = 1000 AND MOTIVO_IA = 'Quiebre de stock' AND FECHA_VISITA >= DATEADD(day, -30, CURRENT_DATE())"),
    ("Combien avons-nous vendu ce mois-ci chez ISM Haïti, par canal ?",
     f"SELECT LISTAGG(CANAL_CORPORATIVO || ': USD ' || TO_VARCHAR(ROUND(v), 'FM999,999,999'), ' | ') FROM (SELECT CANAL_CORPORATIVO, SUM(USD_VALOR_NETO) v FROM {fq}.FACT_DETALLE_VENTAS WHERE ID_SOCIEDAD = 3000 AND PERIODO = DATE_TRUNC('month', CURRENT_DATE()) GROUP BY 1 ORDER BY 2 DESC)"),
]

for i, (q, expected_sql) in enumerate(evaluacion, 1):
    esperado = session.sql(expected_sql).collect()[0][0]
    respuesta = preguntar(q)
    print(f"--- Pregunta {i}: {q}\nESPERADO : {esperado}\nAGENTE   : {respuesta[:400]}\n")

print("Ahora usted: ¿en cuántas acertó el agente? Si alguna falla, pregúntele a Cortex Code qué le falta al modelo semántico.")

Esto es lo mínimo para llevar el agente a otros países: **un set de preguntas con respuestas esperadas** que se vuelve a correr cada vez que cambia el modelo semántico. Snowflake también ofrece [evaluaciones de Cortex Agents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-evaluations) para hacerlo de forma sistemática.

## Lo que se lleva

1. **Cortex Code acelera, usted decide:** cada paso empezó con un prompt de negocio y terminó con SQL que usted revisó.
2. **Las AI Functions convierten texto libre en datos**, dentro de Snowflake y con un costo controlado gracias al refresco incremental.
3. **Valide la IA con datos:** tres errores reales, medidos y llevados a cero con mejor contexto y reglas determinísticas.
4. **La definición de negocio vive en el modelo semántico**, no en el prompt. Así Comercial y Finanzas reportan la misma cifra.
5. **Un agente se evalúa antes de escalarlo a otros países.**

---
## ★ Reto opcional avanzado (para quienes terminen antes)

Supply quiere saber si el quiebre de Rayo es un problema de inventario. Extienda el agente para que responda:

> *"¿Cuántos días de cobertura de producto terminado tiene cada sociedad?"*

**Lo que tiene que lograr:**
1. **Extender** `SV_COMERCIAL_ISM`, sin perder las notas ni la corrección de ventas, con la tabla `INVENTARIOS_MB52`.
2. Crear una métrica llamada exactamente **`DIAS_COBERTURA`** = cajas de **producto terminado** en el **último corte** de inventario ÷ venta diaria promedio en cajas del **mes actual** (cajas del mes ÷ días del mes).
3. Comprobar que el agente la usa: *"¿Qué sociedad tiene menos días de cobertura?"*.

**Pistas:** pídale a Cortex Code que use `DESCRIBE SEMANTIC VIEW` para ver la definición actual. Una métrica que combina dos tablas se define como *derived metric*, a partir de una métrica de cada tabla.

Cuando crea que lo tiene, ejecute el validador:

In [ ]:
def validar_reto_lab2():
    """Valida el reto opcional. No modifica nada."""
    problemas = []
    desc = session.sql(f"DESCRIBE SEMANTIC VIEW {fq}.SV_COMERCIAL_ISM").collect()
    tablas = {r["property_value"] for r in desc if r["property"] == "BASE_TABLE_NAME"}
    metricas = {r["object_name"] for r in desc if r["object_kind"] in ("METRIC", "DERIVED_METRIC")}
    if "INVENTARIOS_MB52" not in tablas:
        problemas.append("La semantic view todavía no incluye la tabla INVENTARIOS_MB52.")
    if "NOTAS_ENRIQUECIDAS" not in tablas:
        problemas.append("La semantic view perdió la tabla de notas: extiéndala, no la reemplace.")
    if "DIAS_COBERTURA" not in metricas:
        problemas.append("No existe una métrica llamada DIAS_COBERTURA.")
    if not problemas:
        esperado = {r[0]: float(r[1]) for r in session.sql(f"""WITH inv AS (
  SELECT SOCIEDAD AS ID_SOCIEDAD, SUM(STOCK_LIB_UTIL) AS CAJAS_INVENTARIO
  FROM {fq}.INVENTARIOS_MB52 WHERE ES_ULTIMO_CORTE AND TIPO_MATERIAL = 'Producto terminado' GROUP BY 1),
vta AS (
  SELECT ID_SOCIEDAD, SUM(CANTIDAD_CAJAS) / DAY(LAST_DAY(CURRENT_DATE())) AS CAJAS_DIA
  FROM {fq}.FACT_DETALLE_VENTAS WHERE PERIODO = DATE_TRUNC('month', CURRENT_DATE()) GROUP BY 1)
SELECT s.SOCIEDAD, ROUND(inv.CAJAS_INVENTARIO / vta.CAJAS_DIA, 1) AS DIAS_COBERTURA
FROM inv JOIN vta USING (ID_SOCIEDAD) JOIN {fq}.DIM_SOCIEDAD s USING (ID_SOCIEDAD)""").collect()}
        actual = {r[0]: float(r[1]) for r in session.sql(
            f"SELECT * FROM SEMANTIC_VIEW({fq}.SV_COMERCIAL_ISM DIMENSIONS sociedades.sociedad METRICS dias_cobertura)").collect()}
        for soc, val in esperado.items():
            if soc not in actual or abs(actual[soc] - val) > 0.5:
                problemas.append(f"{soc}: esperado {val:.1f} días, su métrica da {actual.get(soc, 'nada')}.")
        if session.sql(f"SELECT ROUND(SUM(USD_VALOR_NETO)) FROM {fq}.FACT_DETALLE_VENTAS WHERE PERIODO = DATE_TRUNC('month', CURRENT_DATE())").collect()[0][0] != \
           session.sql(f"SELECT ROUND(ventas) FROM SEMANTIC_VIEW({fq}.SV_COMERCIAL_ISM METRICS ventas.ventas WHERE ventas.es_mes_actual)").collect()[0][0]:
            problemas.append("La métrica 'ventas' ya no es la venta neta: no pierda la corrección de la sección 7.")
    if problemas:
        print("Todavía no:\n- " + "\n- ".join(problemas))
    else:
        acct = session.sql("SELECT CURRENT_ACCOUNT_NAME(), CURRENT_TIMESTAMP()::STRING").collect()[0]
        print("=" * 60)
        print("  RETO DEL LAB 2 COMPLETADO")
        print(f"  Cuenta: {acct[0]}   Hora: {acct[1][:19]}")
        print("  ¡Levante la mano!")
        print("=" * 60)

validar_reto_lab2()

---
### Limpieza (opcional)
Su cuenta de lab se elimina al final del evento. Si quiere borrar los objetos antes, quite el `--` y ejecute la celda.

In [ ]:
%%sql -r limpieza
-- DROP SCHEMA IF EXISTS {{fq}};